## Construct a fully labeled dataset


Final Output: 
- **Rows**: Subjects  
- **Columns**:  
  - `participant_id`  
  - `Age` (float)  
  - `Gender` (‘M’ or ‘F’)  
  - `Adult` (1 or 0)  
  - Connectivity features with names like `"Fusiform gyrus - Insula antero-superior"`  

#### **Cell 1: Imports**

In [3]:
import numpy as np
import pandas as pd
from nilearn import datasets
from math import comb

#### **Cell 2: Generate Feature Name Mapping (DiFuMo-64)**

In [4]:
# Parameters
DIM = 64
FEATURE_NAMES_CSV = "development_fmri_feature_names.csv"

# Fetch DiFuMo atlas
print("📥 Fetching DiFuMo-64 atlas...")
difumo = datasets.fetch_atlas_difumo(dimension=DIM, resolution_mm=2)
labels = difumo.labels["difumo_names"].tolist()

# Generate region-pair names (upper triangle, i < j)
print("⚙️ Generating feature names...")
feature_names = [
    f"{labels[i]} - {labels[j]}"
    for i in range(DIM)
    for j in range(i + 1, DIM)
]

# Validate
n_features = len(feature_names)
assert n_features == comb(DIM, 2) == 2016, f"Expected 2016 features, got {n_features}."

# Save mapping
feature_mapping_df = pd.DataFrame({
    "Feature_Index": range(1, n_features + 1),
    "Region_Pair": feature_names
})
feature_mapping_df.to_csv(FEATURE_NAMES_CSV, index=False)
print(f"✅ Feature names saved to '{FEATURE_NAMES_CSV}'")

📥 Fetching DiFuMo-64 atlas...


[fetch_atlas_difumo] Dataset found in /home/jaizor/nilearn_data/difumo_atlases

⚙️ Generating feature names...
✅ Feature names saved to 'development_fmri_feature_names.csv'


#### **Cell 3: Load Connectivity Features and Metadata**

In [8]:
# Load precomputed data
NPZ_FILE = "development_fmri_connectivity_features.npz"
npz = np.load(NPZ_FILE, allow_pickle=True)

X = npz['X']                     # (155, 2016)
subject_ids = npz['subject_ids'] # e.g., ['sub-pixar001', ...]
labels_struct = npz['labels']    # structured array: Child_Adult, Age, Gender

print(f"✅ Loaded feature matrix: {X.shape}")
print(f"✅ Sample subject IDs: {subject_ids[:3]}")

✅ Loaded feature matrix: (155, 2016)
✅ Sample subject IDs: ['sub-pixar155' 'sub-pixar123' 'sub-pixar124']


#### **Cell 4: Build Phenotypic DataFrame with Binary 'Adult' Label**

In [9]:
# Convert structured array to DataFrame
pheno = pd.DataFrame(labels_struct)

# Insert subject IDs
pheno.insert(0, 'participant_id', subject_ids)

# Encode 'Child_Adult' → binary 'Adult' (1 = adult, 0 = child)
pheno['Adult'] = (pheno['Child_Adult'] == 'adult').astype(int)

# Drop the original categorical column
pheno = pheno.drop(columns=['Child_Adult'])

print("📊 Phenotypic columns:", pheno.columns.tolist())
display(pheno.head())

📊 Phenotypic columns: ['participant_id', 'Age', 'Gender', 'Adult']


,participant_id,Age,Gender,Adult
0,sub-pixar155,26.00,M,1
1,sub-pixar123,27.06,F,1
2,sub-pixar124,33.44,M,1
3,sub-pixar125,31.00,M,1
4,sub-pixar126,19.00,F,1


#### **Cell 5: Merge Phenotype and Connectivity into One DataFrame**

In [10]:
# Load feature names
feature_columns = feature_mapping_df['Region_Pair'].tolist()
assert len(feature_columns) == X.shape[1], "Feature count mismatch."

# Create connectivity DataFrame
conn_df = pd.DataFrame(X, columns=feature_columns, index=subject_ids).reset_index()
conn_df.rename(columns={'index': 'participant_id'}, inplace=True)

# Merge with phenotype on participant_id
final_df = pd.merge(pheno, conn_df, on='participant_id', how='inner')

print("✅ Final dataset shape:", final_df.shape)
print("✅ Columns include:", list(final_df.columns[:5]) + ["..."])

✅ Final dataset shape: (155, 2020)
✅ Columns include: ['participant_id', 'Age', 'Gender', 'Adult', 'Superior frontal sulcus - Fusiform gyrus', '...']


#### **Cell 6: Save Comprehensive Dataset**

In [13]:
OUTPUT_CSV = "development_fmri_full_df.csv"
final_df.to_csv(OUTPUT_CSV, index=False)
print(f"💾 Full labeled dataset saved to: {OUTPUT_CSV}")

# Optional: Show first few columns of output
final_df.head()

💾 Full labeled dataset saved to: development_fmri_full_df.csv


,participant_id,Age,Gender,Adult,Superior frontal sulcus - Fusiform gyrus,Superior frontal sulcus - Calcarine cortex posterior,Superior frontal sulcus - Cingulate cortex posterior,Superior frontal sulcus - Parieto-occipital sulcus superior,Superior frontal sulcus - Insula antero-superior,Superior frontal sulcus - Superior temporal sulcus with angular gyrus,...,Cuneus - Middle temporal gyrus,Cuneus - Superior frontal gyrus,Cuneus - Central sulcus,Cuneus - Caudate,Middle temporal gyrus - Superior frontal gyrus,Middle temporal gyrus - Central sulcus,Middle temporal gyrus - Caudate,Superior frontal gyrus - Central sulcus,Superior frontal gyrus - Caudate,Central sulcus - Caudate
0,sub-pixar155,26.00,M,1,0.212227,-0.122411,-0.007993,0.249761,-0.162396,-0.110983,...,0.056761,-0.094024,-0.049317,0.394151,0.338249,0.192816,-0.104195,0.306159,0.293067,-0.119750
1,sub-pixar123,27.06,F,1,-0.101834,-0.153817,0.371161,0.452083,-0.077127,-0.238911,...,0.151787,0.073846,0.034794,-0.026017,0.257510,0.083997,0.212111,0.195386,0.212271,0.142490
2,sub-pixar124,33.44,M,1,0.082511,0.004679,0.391922,0.114839,-0.060891,0.013514,...,-0.052295,-0.174094,-0.032563,0.072365,0.414372,0.042002,-0.054918,0.325744,0.149890,0.157104
3,sub-pixar125,31.00,M,1,-0.012880,-0.048787,0.366854,0.348423,0.005094,0.035892,...,0.269777,0.069913,0.082954,0.175521,0.576111,0.279639,-0.023321,0.452959,0.444246,0.173480
4,sub-pixar126,19.00,F,1,0.189183,-0.008560,0.262193,0.328338,-0.115477,-0.098379,...,0.079685,-0.004914,-0.014416,0.205490,0.208832,0.332897,-0.053878,0.477224,-0.101495,0.160329
